# Fine-tune a small LLM for agent tool calling

This trains a LoRA adapter on Qwen2.5-1.5B, serves it with vLLM, and runs the agent's 38-question benchmark on the base and fine-tuned models.




## 1. Check for a GPU

In [1]:
import subprocess, torch
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > T4 GPU, then run this again."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

Tesla T4 15.6 GB


## 2. Save progress to Google Drive

Allow access when asked. Everything goes in one folder named `agent-lora-finetune`.

In [3]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/agent-lora-finetune"
os.makedirs(DRIVE, exist_ok=True)
os.environ["DRIVE"] = DRIVE
print("Saving to", DRIVE)

Mounted at /content/drive
Saving to /content/drive/MyDrive/agent-lora-finetune


## 3. Install the project and the agent

The agent is pinned to the commit that produced the Gemini score, so all the models get the same prompt, tools and benchmark.

In [4]:
%cd /content
!rm -rf agent-lora-finetune
!git clone -q https://github.com/ykrubha/agent-lora-finetune.git
%cd /content/agent-lora-finetune
!git clone -q https://github.com/ykrubha/agentic-data-analyst.git agent
!git -C agent checkout -q e82e4f7a52f01fe96d8d9d37a49a8ae1161fc8fa
!pip install -q -e agent -e . pytest "transformers>=4.56" "peft>=0.12" "accelerate>=0.33"
!git -C agent log -1 --format='Agent commit: %h %s'
import sys
# This kernel started before the install, so add the new packages to its path directly.
sys.path[:0] = ["/content/agent-lora-finetune/src", "/content/agent-lora-finetune/agent/src"]


/content
/content/agent-lora-finetune
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for analyst-agent (pyproject.toml) ... done
  Building editable for agent-lora-finetune (pyproject.toml) ... done
Agent commit: e82e4f7 Add benchmark results


## 4. Generate the training data

This runs the agent's real tools on 8 synthetic databases. None of them are the benchmark database, and two are held out for validation.

In [ ]:
!pip install -q "sqlglot==30.29.0"
!pip uninstall -y -q torchao

In [5]:
!pip install -q "sqlglot==30.29.0"
!python scripts/generate_data.py

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires sqlglot<25.21,>=23.4, but you have sqlglot 30.29.0 which is incompatible.
{
 "train": 505,
 "validation": 62,
 "skipped": 2,
 "train_domains": {
  "logistics": 89,
  "airline": 78,
  "retail": 111,
  "energy": 75,
  "university": 60,
  "hospital": 92
 },
 "train_kinds": {
  "multi_step": 84,
  "aggregation": 181,
  "join": 102,
  "python": 84,
  "refusal": 54
 },
 "validation_kinds": {
  "refusal": 6,
  "multi_step": 10,
  "join": 14,
  "aggregation": 22,
  "python": 10
 }
}


## 5. Check the real tokenizer before training

This stops here if Qwen's chat template breaks the training labels. The printed text should show only tool calls, with no system prompt or tool results.

In [6]:
import json, sys
import numpy as np
from transformers import AutoTokenizer
from finetune.train import IGNORE, encode_example, load_jsonl

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
rows = load_jsonl("data/train.jsonl")
encoded = [encode_example(row, tokenizer, 100000) for row in rows]
lengths = [len(item["input_ids"]) for item in encoded]
share = np.mean([np.mean(np.array(item["labels"]) != IGNORE) for item in encoded])
print(f"{len(rows)} examples. Tokens: median {int(np.median(lengths))}, 95th percentile {int(np.percentile(lengths, 95))}, longest {max(lengths)}.")
print(f"Over the 3072 token limit: {sum(n > 3072 for n in lengths)}. Share of tokens used as targets: {share:.1%}.")

sample = encoded[0]
targets = tokenizer.decode([i for i, label in zip(sample["input_ids"], sample["labels"]) if label != IGNORE])
print("\n--- Training targets for one example ---\n" + targets[:1200])
assert "<tool_call>" in targets and "<tool_response>" not in targets and "careful data analyst" not in targets
print("\nTokenizer check passed.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

505 examples. Tokens: median 1636, 95th percentile 2018, longest 2234.
Over the 3072 token limit: 0. Share of tokens used as targets: 9.6%.

--- Training targets for one example ---
<tool_call>
{"name": "list_tables", "arguments": {}}
</tool_call><|im_end|>
<tool_call>
{"name": "describe_table", "arguments": {"table": "Shipment"}}
</tool_call><|im_end|>
<tool_call>
{"name": "run_sql", "arguments": {"query": "SELECT strftime('%Y-%m', \"ShipDate\") AS label, SUM(\"Charge\") AS metric FROM \"Shipment\" GROUP BY label ORDER BY metric DESC LIMIT 2", "save_as": "df"}}
</tool_call><|im_end|>
<tool_call>
{"name": "final_answer", "arguments": {"answer": "Total charge was highest in 2024-04.", "value": "2024-04", "answerable": true, "assumptions": []}}
</tool_call><|im_end|>


Tokenizer check passed.


In [7]:
!pip uninstall -y -q torchao

## 6. Train

This should take about 20 to 40 minutes on a T4. Paste the printed `validation_loss` lines when you share results.

In [8]:
!python scripts/train_lora.py --model Qwen/Qwen2.5-1.5B-Instruct --output "$DRIVE/adapter" --epochs 3


model.safetensors: downloading bytes:   0% 0.00/3.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   3% 87.9M/3.09G [00:01<00:32, 92.1MB/s, 7.08MB/s  ]
model.safetensors: downloading bytes:   6% 181M/3.09G [00:02<00:21, 133MB/s, 15.0MB/s  ]
model.safetensors: downloading bytes:   6% 196M/3.09G [00:02<00:21, 135MB/s, 16.3MB/s  ]
model.safetensors: downloading bytes:   7% 212M/3.09G [00:02<00:20, 141MB/s, 17.5MB/s  ]
model.safetensors: downloading bytes:   7% 231M/3.09G [00:02<00:18, 155MB/s, 18.9MB/s  ]
model.safetensors: reconstructing file:   2% 48.5M/3.09G [00:02<00:56, 54.1MB/s,  795kB/s  ]
model.safetensors: downloading bytes:   8% 257M/3.09G [00:02<00:19, 142MB/s, 21.6MB/s  ]
model.safetensors: downloading bytes:   9% 285M/3.09G [00:03<00:24, 116MB/s, 23.1MB/s  ]
model.safetensors: downloading bytes:  10% 300M/3.09G [00:03<00:27, 100MB/s, 24.9MB/s  ]
model.safetensors: reconstructing file:   8% 232M/3.09G [00:03<00:15, 186MB/s, 18.7MB/s  ]
model.safetensors: downloading 

## 7. Start vLLM with both models

vLLM gets its own environment so it can't conflict with Colab's packages. This takes several minutes the first time, and the server keeps running in the background.

In [12]:
!pip install -q uv
!rm -rf /content/vllm-env
!uv venv --python 3.12 /content/vllm-env
!uv pip install --python /content/vllm-env/bin/python vllm 2>&1 | tail -n 15
!ls -la /content/vllm-env/bin/vllm
!/content/vllm-env/bin/python -c "import vllm; print('vLLM', vllm.__version__)"

Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /content/vllm-env
Activate with: source /content/vllm-env/bin/activate
 + transformers==5.20.2
 + triton==3.8.0
 + truststore==0.10.5
 + typer==0.27.3
 + typing-extensions==4.16.0
 + typing-inspection==0.4.5
 + urllib3==2.8.0
 + uvicorn==0.54.0
 + uvloop==0.23.0
 + vllm==0.33.0
 + watchfiles==1.3.0
 + websockets==17.4
 + xgrammar==0.2.8
 + yarl==1.25.1
 + z3-solver==4.15.4.0
-rwxr-xr-x 1 root root 319 Nov  5 16:19 /content/vllm-env/bin/vllm
vLLM 0.33.0


In [13]:
import subprocess, time, urllib.request, json, os
log = open("/content/vllm.log", "w")
server = subprocess.Popen([
    "/content/vllm-env/bin/vllm", "serve", "Qwen/Qwen2.5-1.5B-Instruct",
    "--served-model-name", "base",
    "--enable-lora", "--max-lora-rank", "16",
    "--lora-modules", f"analyst={os.environ['DRIVE']}/adapter/best",
    "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
    "--max-model-len", "8192", "--gpu-memory-utilization", "0.85", "--dtype", "float16",
    "--override-generation-config", json.dumps({"temperature": 0.0}),
    "--port", "8000",
], stdout=log, stderr=subprocess.STDOUT)

for _ in range(120):
    if server.poll() is not None:
        break
    try:
        reply = json.load(urllib.request.urlopen("http://localhost:8000/v1/models", timeout=3))
        print("Ready:", [m["id"] for m in reply["data"]])
        break
    except Exception:
        time.sleep(5)
else:
    print("Timed out.")
if server.poll() is not None or not os.path.exists("/content/vllm.log"):
    print("The server stopped. Paste this log when asking for help:")
print(subprocess.run(["tail", "-n", "15", "/content/vllm.log"], capture_output=True, text=True).stdout)

Ready: ['base', 'analyst']
(APIServer pid=13163) INFO 11-05 16:25:40 [launcher.py:197] Route: /generative_scoring, Methods: POST
(APIServer pid=13163) INFO 11-05 16:25:40 [launcher.py:197] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=13163) INFO 11-05 16:25:40 [launcher.py:197] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=13163) INFO 11-05 16:25:40 [launcher.py:197] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=13163) INFO 11-05 16:25:40 [launcher.py:197] Route: /v1/messages/render, Methods: POST
(APIServer pid=13163) INFO 11-05 16:25:40 [launcher.py:197] Route: /v1/completions/render, Methods: POST
(APIServer pid=13163) INFO 11-05 16:25:40 [launcher.py:197] Route: /v1/responses/render, Methods: POST
(APIServer pid=13163) INFO 11-05 16:25:40 [launcher.py:197] Route: /v1/chat/completions/derender, Methods: POST
(APIServer pid=13163) INFO 11-05 16:25:40 [launcher.py:197] Route: /v1/completions/derender, Methods: POST
(APIServer pid=13163) INFO 

## 8. Try one step on each model

A good result prints a tool call such as `list_tables`. The base model may print plain text instead, which is one of the failures we're measuring.

In [14]:
import os, sys
os.environ["OPENAI_BASE_URL"] = "http://localhost:8000/v1"
from analyst_agent.agent import SYSTEM_PROMPT
from analyst_agent.llm import make_provider
from analyst_agent.tools import TOOL_SCHEMAS

for name in ("base", "analyst"):
    reply = make_provider("openai", name).complete(
        SYSTEM_PROMPT, [{"role": "user", "content": "Which genre generated the most revenue?"}], TOOL_SCHEMAS)
    print(f"{name}: tool calls = {reply.tool_calls}\n  text = {reply.text[:200]!r}\n  output tokens = {reply.output_tokens}\n")

base: tool calls = [ToolCall(id='chatcmpl-tool-aca7c1572315a4fb', name='list_tables', arguments={}, extra_content=None), ToolCall(id='chatcmpl-tool-8aedf947f862b20a', name='describe_table', arguments={'table': 'genres_revenue'}, extra_content=None)]
  text = ''
  output tokens = 38

analyst: tool calls = [ToolCall(id='chatcmpl-tool-82bc1cd4f98cd79d', name='list_tables', arguments={}, extra_content=None)]
  text = ''
  output tokens = 16



## 9. Run the benchmark on both models

This should take about 30 to 90 minutes. Each question prints `PASS` or `FAIL`. If Colab disconnects, reconnect and run everything again, and `--resume` will skip the finished questions.

Failures marked `ERROR` are server problems, not wrong answers, and no score gets written while any remain.

In [15]:
%cd /content/agent-lora-finetune/agent
!python scripts/download_data.py
!OPENAI_BASE_URL=http://localhost:8000/v1 python -m eval.run_eval --provider openai --model base --max-seconds 600 --resume --out "$DRIVE/results/base"

/content/agent-lora-finetune/agent
OK: data/chinook.sqlite
FAIL agg01  did not finish: no_final_answer
FAIL agg02  expected 2328.6, got None
FAIL agg03  expected 5.651941747572815, got '$789.60'
FAIL agg04  expected 260, got None
FAIL agg05  did not finish: too_many_errors
FAIL agg06  did not finish: too_many_errors
FAIL agg07  expected 469.58, got None
FAIL agg08  expected 1010.4619407653809, got '192'
FAIL join01 expected 'Iron Maiden', got None
FAIL join02 did not finish: no_final_answer
FAIL join03 did not finish: too_many_errors
FAIL join04 expected 3 items, got 'John Doe, Jane Smith, Alex Johnson'
FAIL join05 expected 'Jane Peacock', got None
FAIL join06 did not finish: too_many_errors
FAIL join07 expected 105.92999999999999, got None
FAIL join08 expected 'Music', got 'Most Tracks'
FAIL join09 expected 1519, got None
FAIL join10 did not finish: no_final_answer
FAIL mul01  expected '2022-01', got None
FAIL mul02  expected 77.52469294855278, got '60%'
FAIL mul03  expected 22, got N

In [17]:
import json, collections
data = json.load(open(f"{DRIVE}/results/base/benchmark.json"))
errors = collections.Counter()
for row in data["rows"]:
    for event in row["trace"]:
        if event["is_error"]:
            errors[f'{event["tool"]}: {event["result"][:110]}'] += 1
print("MOST COMMON ERRORS")
for text, count in errors.most_common(12):
    print(count, text)
for question_id in ("agg01", "agg02", "join01"):
    row = next(r for r in data["rows"] if r["id"] == question_id)
    print(f"\n=== {question_id}: {row['question']} | {row['status']} | value={row['value']!r}")
    print("Answer:", (row["answer"] or "")[:300])
    for event in row["trace"]:
        print(f'[{event["step"]}] {event["tool"]} {json.dumps(event["arguments"])[:220]}')
        print("    ->", "ERROR" if event["is_error"] else "ok", event["result"][:170])

MOST COMMON ERRORS
16 run_sql: Bad arguments for run_sql: ToolExecutor._tool_run_sql() got an unexpected keyword argument 'sql'
6 describe_table: SqlExecutionError: Unknown table 'tracks'. Available: Album, Artist, Customer, Employee, Genre, Invoice, Invoi
5 describe_table: SqlExecutionError: Unknown table 'customers'. Available: Album, Artist, Customer, Employee, Genre, Invoice, In
4 run_sql: SqlExecutionError: no such table: invoices
4 run_sql: SqlExecutionError: no such table: customers
4 run_sql: SqlExecutionError: no such table: sales
3 describe_table: SqlExecutionError: Unknown table 'invoices'. Available: Album, Artist, Customer, Employee, Genre, Invoice, Inv
2 run_sql: SqlExecutionError: no such table: invoice_line
2 run_sql: SqlExecutionError: no such column: T1.ArtistId
2 run_sql: SqlExecutionError: no such table: songs
1 run_sql: SqlExecutionError: no such table: financials
1 describe_table: SqlExecutionError: Unknown table 'artists_albums'. Available: Album, Artist, Custome

In [16]:
!OPENAI_BASE_URL=http://localhost:8000/v1 python -m eval.run_eval --provider openai --model analyst --max-seconds 600 --resume --out "$DRIVE/results/analyst"

PASS agg01  match
PASS agg02  match
PASS agg03  match
FAIL agg04  expected 260, got 0
PASS agg05  match
PASS agg06  match
PASS agg07  match
FAIL agg08  expected 1010.4619407653809, got 1059546140
PASS join01 match
PASS join02 match
PASS join03 match
FAIL join04 refused an answerable question
FAIL join05 expected 'Jane Peacock', got 'Helena'
FAIL join06 expected 213, got 3503
FAIL join07 expected 105.92999999999999, got [1284.03, 11.88]
FAIL join08 expected 'Music', got 'TV Shows'
FAIL join09 expected 1519, got 3503
FAIL join10 expected 'MPEG audio file', got 'Balls to the Wall'
PASS mul01  match
FAIL mul02  expected 77.52469294855278, got 100
FAIL mul03  expected 22, got 179
FAIL mul04  did not finish: too_many_errors
FAIL mul05  did not finish: too_many_errors
FAIL mul06  expected 234.62039660056658, got 6.866666666666666
FAIL mul07  expected 166, got 0
PASS mul08  match
PASS py01   match
PASS py02   match
FAIL py03   refused an answerable question
FAIL py04   refused an answerable qu

## 10. Compare the results

This includes the earlier Gemini score from the agent's repo. Copy the printed table into the chat, and download the zip file for the repo.

In [18]:
%cd /content/agent-lora-finetune
!python scripts/compare.py --run "Qwen 1.5B base=$DRIVE/results/base/benchmark.json" --run "Qwen 1.5B + LoRA=$DRIVE/results/analyst/benchmark.json" --run "Gemini 3.5 Flash Lite=agent/results/benchmark.json" --out "$DRIVE/results/comparison.md"
!cp "$DRIVE/adapter/training_report.json" "$DRIVE/results/" 2>/dev/null
!cp data/summary.json "$DRIVE/results/training_data_summary.json"
!cp /content/vllm.log "$DRIVE/results/vllm.log"
!cd "$DRIVE" && rm -f results.zip && zip -qr results.zip results adapter/best/adapter_config.json adapter/best/adapter_model.safetensors -x "results/vllm.log"
!ls -la "$DRIVE/results.zip"
from google.colab import files
files.download(f"{DRIVE}/results.zip")

/content/agent-lora-finetune
| Metric | Qwen 1.5B base | Qwen 1.5B + LoRA | Gemini 3.5 Flash Lite |
|---|---:|---:|---:|
| Model | `base` | `analyst` | `gemini-3.5-flash-lite` |
| **Correct** | **0/38** | **17/38** | **37/38** |
| aggregation | 0/8 | 6/8 | 8/8 |
| join | 0/10 | 3/10 | 10/10 |
| multi_step | 0/8 | 2/8 | 8/8 |
| python | 0/6 | 3/6 | 5/6 |
| refusal | 0/6 | 3/6 | 6/6 |
| Finished with an answer | 28/38 | 36/38 | 37/38 |
| Answers sent as native tool calls | 28/38 | 36/38 | 37/38 |
| Answers recovered from plain text | 0 | 0 | 0 |
| Tool errors | 77 | 14 | 20 |
| Mean steps | 4.05 | 4.79 | 6.5 |
| Mean seconds per question | 5.2 | 8.2 | 92.9 |
| How runs ended | answered: 28, no_final_answer: 5, too_many_errors: 5 | answered: 36, too_many_errors: 2 | answered: 37, max_steps: 1 |

-rw------- 1 root root 16125151 Nov  5 16:41 /content/drive/MyDrive/agent-lora-finetune/results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>